In [ ]:
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression

brfss_final= pd.read_csv("/content/brfss_final_harmonized.csv")
uganda_final=pd.read_csv("/content/uganda_final_harmonized.csv")
geda_final=pd.read_csv("/content/geda_final_harmonized.csv")

In [ ]:
#Define features
features =["age_group", "sex_male", "bmi_group", "hypertension", "current_smoker", "physically_active"]
target ="diabetes_binary"

In [ ]:
#Function
def run_external_validation(train_df, test_df, train_name, test_name):
    train_data=train_df.dropna(subset=[target]).copy()
    test_data=test_df.dropna(subset=[target]).copy()

    x_train = train_data[features]
    y_train =train_data[target]

    x_test = test_data[features]
    y_test =test_data[target]


    preprocessor =ColumnTransformer(
         transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="most_frequent"))]), features)

         ]

    )

    model = Pipeline([
       ("prep", preprocessor),
       ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))

    ])

    model.fit(x_train, y_train)

    proba =model.predict_proba(x_test)[:,1]
    pred=(proba >=0.5).astype(int)

    results={
        "Train": train_name,
        "Test": test_name,
        "AUC": roc_auc_score(y_test, proba),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0)
    }

    return results

In [ ]:
#External validation

geda_to_uganda =run_external_validation(geda_final, uganda_final, "GEDA", "Uganda")
uganda_to_geda =run_external_validation(uganda_final, geda_final, "Uganda", "GEDA")

external_results =pd.DataFrame([geda_to_uganda, uganda_to_geda])
external_results

,Train,Test,AUC,Precision,Recall,F1
0,GEDA,Uganda,0.655403,0.033028,0.391304,0.060914
1,Uganda,GEDA,0.780674,0.135882,0.931166,0.237156


**GEDA -> Uganda**

**AUC =0.655** maens the model trained on Germany only moderately separates diabetes vs non-diabetes in Uganda.

**Precision =0.033** is very low so many predicted positives are false positives.

**Recall=0.391** means that it catches only about 39% of Uganda diabetes cases.

**Interpretation:** A model trained on German survey data does not transfer well to the Ugandan population.

**This supports my thesis idea that:**

-internal validation is not enough

-population contect matter

-a model that works in one setting may not work well in another

**Uganda -> GEDA**

AUC=0.781 is fairly strong, Recall=0.931 is very high, Precision=0.136 is still low but much better than GEDA ->Uganda, F1=0.237 is still modest but stronger than the reverse transfer.

**Interpretation**

A model trained on Uganda transfers better to Germany than the German-trained model transfers to Uganda

So now i move to SHAP and these are the questions i want to answer:

1. Which variables are most important in GEDA internally and Uganda internally?

2. Do those important variables change under external validation?

3. Are the explanantions similar across populations or do they shift?